# Portfolio Downside-Risk Early-Warning System


This project develops a machine-learning risk scanner using approximately
500 U.S. stocks and historical daily market data from 2010 to 2026.

The objective is to predict whether an individual stock is at elevated risk
of declining by at least 3% during the next five trading days.

The system is designed to help portfolio managers and risk analysts identify
securities that require closer investigation. It is a decision-support and
educational project, not investment advice.

In [ ]:
# Install the market-data package
!pip install yfinance -q

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
import requests
from io import StringIO

url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

# Make the request appear like a normal browser request
headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(
    url,
    headers=headers,
    timeout=30
)

# Confirm that the webpage loaded successfully
response.raise_for_status()

# Read the tables from the downloaded webpage
tables = pd.read_html(StringIO(response.text))

# Select the first table
companies = tables[0]

# Rename the columns
companies = companies.rename(
    columns={
        "Symbol": "Ticker",
        "Security": "Company",
        "GICS Sector": "Sector",
        "GICS Sub-Industry": "Sub_Industry"
    }
)

# Convert tickers such as BRK.B into Yahoo Finance format BRK-B
companies["Ticker"] = companies["Ticker"].str.replace(
    ".",
    "-",
    regex=False
)

print("Number of securities:", len(companies))

companies[
    ["Ticker", "Company", "Sector", "Sub_Industry"]
].head(10)

Number of securities: 503


,Ticker,Company,Sector,Sub_Industry
0,MMM,3M,Industrials,Industrial Conglomerates
1,AOS,A. O. Smith,Industrials,Building Products
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment
3,ABBV,AbbVie,Health Care,Biotechnology
4,ACN,Accenture,Information Technology,IT Consulting & Other Services
5,ADBE,Adobe Inc.,Information Technology,Application Software
6,AMD,Advanced Micro Devices,Information Technology,Semiconductors
7,AES,AES Corporation,Utilities,Independent Power Producers & Energy Traders
8,AFL,Aflac,Financials,Life & Health Insurance
9,A,Agilent Technologies,Health Care,Life Sciences Tools & Services


In [ ]:
# Select five tickers for a test download
test_tickers = companies["Ticker"].head(5).tolist()

print("Testing:", test_tickers)

test_prices = yf.download(
    tickers=test_tickers,
    start="2010-01-01",
    end="2026-08-31",
    group_by="ticker",
    auto_adjust=False,
    progress=True,
    threads=True
)

print("Download completed.")
print("Dataset shape:", test_prices.shape)

test_prices.head()

Testing: ['MMM', 'AOS', 'ABT', 'ABBV', 'ACN']


[*********************100%***********************]  5 of 5 completed

Download completed.
Dataset shape: (4188, 30)


Ticker     ABBV                                        ACN             \
Price      Open High Low Close Adj Close Volume       Open       High   
Date                                                                    
2010-01-04  NaN  NaN NaN   NaN       NaN    NaN  41.520000  42.200001   
2010-01-05  NaN  NaN NaN   NaN       NaN    NaN  42.099998  42.450001   
2010-01-06  NaN  NaN NaN   NaN       NaN    NaN  42.090000  42.900002   
2010-01-07  NaN  NaN NaN   NaN       NaN    NaN  42.500000  42.900002   
2010-01-08  NaN  NaN NaN   NaN       NaN    NaN  42.509998  43.060001   

Ticker                            ...        MMM                        \
Price             Low      Close  ...        Low      Close  Adj Close   
Date                              ...                                    
2010-01-04  41.500000  42.070000  ...  69.122070  69.414719  42.374981   
2010-01-05  41.980000  42.330002  ...  68.311035  68.979935  42.109562   
2010-01-06  41.810001  42.779999  ...  69.824417  69.958191  42.706734   
2010-01-07  42.240002  42.740002  ...  68.662209  70.008362  42.737385   
2010-01-08  42.189999  42.570000  ...  69.648827  70.501671  43.038521   

Ticker                     ABT                                              \
Price        Volume       Open       High        Low      Close  Adj Close   
Date                                                                         
2010-01-04  3640265  26.000362  26.177889  25.870815  26.129908  18.078800   
2010-01-05  3405012  26.134706  26.134706  25.789249  25.918797  17.932737   
2010-01-06  6301126  25.880411  26.096321  25.837231  26.062737  18.032322   
2010-01-07  5346240  26.057938  26.283443  25.942785  26.278646  18.181704   
2010-01-08  4073337  26.273848  26.508949  26.235464  26.412991  18.274664   

Ticker                
Price         Volume  
Date                  
2010-01-04  10829095  
2010-01-05  10562109  
2010-01-06  11401417  
2010-01-07  12857232  
2010-01-08  12148604  

[5 rows x 30 columns]

In [ ]:
import time

# Complete list of securities
all_tickers = companies["Ticker"].tolist()

batch_size = 25
downloaded_data = []
failed_tickers = []

for start_position in range(0, len(all_tickers), batch_size):

    batch = all_tickers[
        start_position:start_position + batch_size
    ]

    batch_number = (start_position // batch_size) + 1
    total_batches = int(np.ceil(len(all_tickers) / batch_size))

    print(
        f"Downloading batch {batch_number} of {total_batches}"
    )

    try:
        batch_prices = yf.download(
            tickers=batch,
            start="2010-01-01",
            end="2026-08-31",
            group_by="ticker",
            auto_adjust=False,
            progress=False,
            threads=True
        )

        available_tickers = (
            batch_prices.columns.get_level_values(0).unique()
        )

        for ticker in batch:

            if ticker not in available_tickers:
                failed_tickers.append(ticker)
                continue

            ticker_data = batch_prices[ticker].copy()

            # Remove completely empty records
            ticker_data = ticker_data.dropna(
                how="all"
            )

            if ticker_data.empty:
                failed_tickers.append(ticker)
                continue

            ticker_data = ticker_data.reset_index()
            ticker_data["Ticker"] = ticker

            downloaded_data.append(ticker_data)

    except Exception as error:
        print("Batch error:", error)
        failed_tickers.extend(batch)

    # Brief pause to reduce rate-limit problems
    time.sleep(2)

print("\nDownload process finished.")
print("Downloaded ticker tables:", len(downloaded_data))
print("Failed tickers:", failed_tickers)


Download process finished.
Downloaded ticker tables: 503
Failed tickers: []


In [ ]:
market_data = pd.concat(
    downloaded_data,
    ignore_index=True
)

# Remove the yfinance column label
market_data.columns.name = None

print("Combined dataset shape:", market_data.shape)
print("Unique securities:", market_data["Ticker"].nunique())
print("Earliest date:", market_data["Date"].min())
print("Latest date:", market_data["Date"].max())

market_data.head()

Combined dataset shape: (1974106, 8)
Unique securities: 503
Earliest date: 2010-01-04 00:00:00
Latest date: 2026-08-28 00:00:00


,Date,Open,High,Low,Close,Adj Close,Volume,Ticker
0,2010-01-04,69.473244,69.774246,69.122070,69.414719,42.374981,3640265.0,MMM
1,2010-01-05,69.230766,69.590302,68.311035,68.979935,42.109562,3405012.0,MMM
2,2010-01-06,70.133781,70.735786,69.824417,69.958191,42.706734,6301126.0,MMM
3,2010-01-07,69.665550,70.033447,68.662209,70.008362,42.737385,5346240.0,MMM
4,2010-01-08,69.974915,70.501671,69.648827,70.501671,43.038521,4073337.0,MMM


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

market_data.to_parquet(
    "/content/drive/MyDrive/sp500_historical_market_data.parquet",
    index=False
)

companies.to_csv(
    "/content/drive/MyDrive/sp500_company_information.csv",
    index=False
)

print("Dataset checkpoint saved to Google Drive.")

Mounted at /content/drive
Dataset checkpoint saved to Google Drive.


In [ ]:
# Keep the required company information
company_details = companies[
    ["Ticker", "Company", "Sector", "Sub_Industry"]
].copy()

# Join company information to every market record
market_data = market_data.merge(
    company_details,
    on="Ticker",
    how="left",
    validate="many_to_one"
)

# Arrange the observations chronologically
market_data = market_data.sort_values(
    ["Ticker", "Date"]
).reset_index(drop=True)

# Remove duplicate ticker-date records
duplicate_count = market_data.duplicated(
    subset=["Ticker", "Date"]
).sum()

market_data = market_data.drop_duplicates(
    subset=["Ticker", "Date"]
).reset_index(drop=True)

print("Rows after cleaning:", len(market_data))
print("Duplicate records removed:", duplicate_count)
print("Unique companies:", market_data["Ticker"].nunique())
print("Number of sectors:", market_data["Sector"].nunique())

market_data.head()

Rows after cleaning: 1974106
Duplicate records removed: 0
Unique companies: 503
Number of sectors: 11


,Date,Open,High,Low,Close,Adj Close,Volume,Ticker,Company,Sector,Sub_Industry
0,2010-01-04,22.453505,22.625179,22.267525,22.389128,19.772947,3815561.0,A,Agilent Technologies,Health Care,Life Sciences Tools & Services
1,2010-01-05,22.324751,22.331903,22.002861,22.145924,19.558159,4186031.0,A,Agilent Technologies,Health Care,Life Sciences Tools & Services
2,2010-01-06,22.067240,22.174536,22.002861,22.067240,19.488672,3243779.0,A,Agilent Technologies,Health Care,Life Sciences Tools & Services
3,2010-01-07,22.017166,22.045780,21.816881,22.038628,19.463404,3095172.0,A,Agilent Technologies,Health Care,Life Sciences Tools & Services
4,2010-01-08,21.917025,22.067240,21.745352,22.031473,19.457094,3733918.0,A,Agilent Technologies,Health Care,Life Sciences Tools & Services


In [ ]:
missing_summary = (
    market_data.isna()
    .sum()
    .sort_values(ascending=False)
)

missing_summary

,0
Date,0
Open,0
High,0
Low,0
Close,0
Adj Close,0
Volume,0
Ticker,0
Company,0
Sector,0


In [ ]:
rows_before = len(market_data)

market_data = market_data.dropna(
    subset=["Close", "Adj Close"]
).copy()

# Remove any impossible non-positive closing prices
market_data = market_data[
    (market_data["Close"] > 0) &
    (market_data["Adj Close"] > 0)
].copy()

market_data = market_data.reset_index(drop=True)

rows_removed = rows_before - len(market_data)

print("Rows before cleaning:", rows_before)
print("Rows removed:", rows_removed)
print("Rows remaining:", len(market_data))
print("Remaining missing values:", market_data.isna().sum().sum())

Rows before cleaning: 1974106
Rows removed: 0
Rows remaining: 1974106
Remaining missing values: 0


In [ ]:
# Confirm chronological order
market_data = market_data.sort_values(
    ["Ticker", "Date"]
).reset_index(drop=True)

ticker_groups = market_data.groupby("Ticker")

# Historical returns
market_data["Daily_Return"] = ticker_groups[
    "Adj Close"
].pct_change(fill_method=None)

market_data["Return_5D"] = ticker_groups[
    "Adj Close"
].pct_change(5, fill_method=None)

market_data["Return_20D"] = ticker_groups[
    "Adj Close"
].pct_change(20, fill_method=None)

# Moving averages
market_data["SMA_10"] = ticker_groups[
    "Adj Close"
].transform(
    lambda values: values.rolling(10).mean()
)

market_data["SMA_20"] = ticker_groups[
    "Adj Close"
].transform(
    lambda values: values.rolling(20).mean()
)

market_data["SMA_50"] = ticker_groups[
    "Adj Close"
].transform(
    lambda values: values.rolling(50).mean()
)

# Price position relative to moving averages
market_data["Price_to_SMA10"] = (
    market_data["Adj Close"] /
    market_data["SMA_10"] - 1
)

market_data["Price_to_SMA20"] = (
    market_data["Adj Close"] /
    market_data["SMA_20"] - 1
)

market_data["Price_to_SMA50"] = (
    market_data["Adj Close"] /
    market_data["SMA_50"] - 1
)

# Historical volatility
market_data["Volatility_10D"] = ticker_groups[
    "Daily_Return"
].transform(
    lambda values: values.rolling(10).std()
)

market_data["Volatility_20D"] = ticker_groups[
    "Daily_Return"
].transform(
    lambda values: values.rolling(20).std()
)

# Trading-volume signal
market_data["Average_Volume_20D"] = ticker_groups[
    "Volume"
].transform(
    lambda values: values.rolling(20).mean()
)

market_data["Volume_Ratio_20D"] = (
    market_data["Volume"] /
    market_data["Average_Volume_20D"]
)

# Current trading range
market_data["Daily_Price_Range"] = (
    market_data["High"] - market_data["Low"]
) / market_data["Close"]

# Recent drawdown from the 20-day high
market_data["Rolling_High_20D"] = ticker_groups[
    "Adj Close"
].transform(
    lambda values: values.rolling(20).max()
)

market_data["Drawdown_20D"] = (
    market_data["Adj Close"] /
    market_data["Rolling_High_20D"] - 1
)

print("Feature engineering completed.")
print("Dataset shape:", market_data.shape)

Feature engineering completed.
Dataset shape: (1974106, 27)


In [ ]:
# Five-day future return
market_data["Future_Return_5D"] = (
    ticker_groups["Adj Close"].shift(-5) /
    market_data["Adj Close"] - 1
)

# Preserve unavailable future observations as missing
market_data["High_Risk"] = np.where(
    market_data["Future_Return_5D"].notna(),
    (
        market_data["Future_Return_5D"] <= -0.03
    ).astype(int),
    np.nan
)

print(
    market_data[
        ["Ticker", "Date", "Adj Close",
         "Future_Return_5D", "High_Risk"]
    ].tail(10)
)

        Ticker       Date  Adj Close  Future_Return_5D  High_Risk
1974096    ZTS 2026-08-14  73.790001          0.053395        0.0
1974097    ZTS 2026-08-17  72.970001          0.057558        0.0
1974098    ZTS 2026-08-18  73.959999          0.045024        0.0
1974099    ZTS 2026-08-19  76.680000          0.010955        0.0
1974100    ZTS 2026-08-20  75.019997          0.000000        0.0
1974101    ZTS 2026-08-21  77.730003               NaN        NaN
1974102    ZTS 2026-08-24  77.169998               NaN        NaN
1974103    ZTS 2026-08-25  77.290001               NaN        NaN
1974104    ZTS 2026-08-26  77.519997               NaN        NaN
1974105    ZTS 2026-08-27  75.019997               NaN        NaN


In [ ]:
risk_distribution = (
    market_data["High_Risk"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

print("Target distribution:")
print(risk_distribution)

Target distribution:
High_Risk
0.0    84.02
1.0    15.98
Name: proportion, dtype: float64


In [ ]:
indicator_symbols = {
    "SPY": "SPY",
    "VIX": "^VIX",
    "Treasury_Yield": "^TNX",
    "High_Yield_Bonds": "HYG",
    "Long_Term_Treasuries": "TLT"
}

indicator_data = pd.DataFrame()

for indicator_name, symbol in indicator_symbols.items():

    downloaded = yf.download(
        symbol,
        start="2010-01-01",
        end="2026-08-31",
        auto_adjust=False,
        progress=False
    )

    # Handle yfinance multi-level column names
    if isinstance(downloaded.columns, pd.MultiIndex):
        downloaded.columns = downloaded.columns.get_level_values(0)

    price_column = (
        "Adj Close"
        if "Adj Close" in downloaded.columns
        else "Close"
    )

    series = downloaded[[price_column]].copy()
    series.columns = [indicator_name]

    if indicator_data.empty:
        indicator_data = series
    else:
        indicator_data = indicator_data.join(
            series,
            how="outer"
        )

indicator_data = indicator_data.reset_index()

print("Indicator dataset shape:", indicator_data.shape)
indicator_data.head()

Indicator dataset shape: (4190, 6)


,Date,SPY,VIX,Treasury_Yield,High_Yield_Bonds,Long_Term_Treasuries
0,2010-01-04,84.578491,20.040001,3.841,33.888107,55.070801
1,2010-01-05,84.802376,19.350000,3.755,34.048862,55.426468
2,2010-01-06,84.862091,19.160000,3.808,34.136913,54.684505
3,2010-01-07,85.220314,19.059999,3.822,34.274738,54.776508
4,2010-01-08,85.503899,18.129999,3.808,34.328312,54.751961


In [ ]:
indicator_data = indicator_data.sort_values(
    "Date"
).reset_index(drop=True)

# Broad-market performance
indicator_data["SPY_Return_1D"] = (
    indicator_data["SPY"].pct_change()
)

indicator_data["SPY_Return_5D"] = (
    indicator_data["SPY"].pct_change(5)
)

indicator_data["SPY_Volatility_20D"] = (
    indicator_data["SPY_Return_1D"]
    .rolling(20)
    .std()
)

# Changes in market stress
indicator_data["VIX_Change_5D"] = (
    indicator_data["VIX"].pct_change(5)
)

indicator_data["Yield_Change_5D"] = (
    indicator_data["Treasury_Yield"].diff(5)
)

indicator_data["HYG_Return_5D"] = (
    indicator_data["High_Yield_Bonds"].pct_change(5)
)

indicator_data["TLT_Return_5D"] = (
    indicator_data["Long_Term_Treasuries"].pct_change(5)
)

indicator_data.tail()

,Date,SPY,VIX,Treasury_Yield,High_Yield_Bonds,Long_Term_Treasuries,SPY_Return_1D,SPY_Return_5D,SPY_Volatility_20D,VIX_Change_5D,Yield_Change_5D,HYG_Return_5D,TLT_Return_5D
4185,2026-08-24,763.469971,15.85,4.704,79.265594,82.244843,-0.002938,-0.011907,0.008366,0.043450,-0.020,0.001130,0.014874
4186,2026-08-25,765.909973,15.45,4.639,79.484398,83.151375,0.003196,-0.002007,0.008372,-0.024621,-0.067,0.004904,0.022165
4187,2026-08-26,766.080017,15.21,4.664,79.464516,82.982025,0.000222,-0.003875,0.007362,0.021491,0.011,0.002384,0.003373
4188,2026-08-27,771.099976,14.51,4.672,79.434677,82.812668,0.006553,0.011146,0.006637,-0.093691,-0.024,0.003897,0.009594
4189,2026-08-28,769.349976,NaN,NaN,NaN,NaN,-0.002269,0.004741,0.006582,-0.040978,NaN,0.003266,0.013163


In [ ]:
market_data = market_data.merge(
    indicator_data,
    on="Date",
    how="left",
    validate="many_to_one"
)

print("Updated dataset shape:", market_data.shape)

market_data[
    [
        "Ticker",
        "Date",
        "Adj Close",
        "SPY_Return_5D",
        "VIX",
        "Treasury_Yield",
        "High_Risk"
    ]
].head()

Updated dataset shape: (1974106, 41)


,Ticker,Date,Adj Close,SPY_Return_5D,VIX,Treasury_Yield,High_Risk
0,A,2010-01-04,19.772947,NaN,20.040001,3.841,0.0
1,A,2010-01-05,19.558159,NaN,19.350000,3.755,0.0
2,A,2010-01-06,19.488672,NaN,19.160000,3.808,0.0
3,A,2010-01-07,19.463404,NaN,19.059999,3.822,0.0
4,A,2010-01-08,19.457094,NaN,18.129999,3.808,0.0


In [ ]:
feature_columns = [
    "Daily_Return",
    "Return_5D",
    "Return_20D",
    "Price_to_SMA10",
    "Price_to_SMA20",
    "Price_to_SMA50",
    "Volatility_10D",
    "Volatility_20D",
    "Volume_Ratio_20D",
    "Daily_Price_Range",
    "Drawdown_20D",
    "SPY_Return_1D",
    "SPY_Return_5D",
    "SPY_Volatility_20D",
    "VIX",
    "VIX_Change_5D",
    "Treasury_Yield",
    "Yield_Change_5D",
    "HYG_Return_5D",
    "TLT_Return_5D"
]

model_columns = (
    ["Ticker", "Company", "Sector", "Date"]
    + feature_columns
    + ["Future_Return_5D", "High_Risk"]
)

model_data = market_data[
    model_columns
].copy()

# Convert infinite calculations into missing values
model_data = model_data.replace(
    [np.inf, -np.inf],
    np.nan
)

rows_before = len(model_data)

# Remove records that cannot be used by the models
model_data = model_data.dropna(
    subset=feature_columns + ["High_Risk"]
).copy()

model_data["High_Risk"] = (
    model_data["High_Risk"].astype(int)
)

print("Rows before modelling cleanup:", rows_before)
print("Rows available for modelling:", len(model_data))
print("Rows removed:", rows_before - len(model_data))
print("Remaining missing feature values:",
      model_data[feature_columns].isna().sum().sum())

Rows before modelling cleanup: 1974106
Rows available for modelling: 1944165
Rows removed: 29941
Remaining missing feature values: 0


In [ ]:
train_data = model_data[
    model_data["Date"] <= "2022-12-23"
].copy()

validation_data = model_data[
    (model_data["Date"] >= "2023-01-03") &
    (model_data["Date"] <= "2024-12-20")
].copy()

test_data = model_data[
    model_data["Date"] >= "2025-01-02"
].copy()

print("Training rows:", len(train_data))
print("Validation rows:", len(validation_data))
print("Testing rows:", len(test_data))

print("\nTraining period:",
      train_data["Date"].min(),
      "to",
      train_data["Date"].max())

print("Validation period:",
      validation_data["Date"].min(),
      "to",
      validation_data["Date"].max())

print("Testing period:",
      test_data["Date"].min(),
      "to",
      test_data["Date"].max())

print("\nHigh-risk percentage by partition:")

print(
    "Training:",
    round(train_data["High_Risk"].mean() * 100, 2),
    "%"
)

print(
    "Validation:",
    round(validation_data["High_Risk"].mean() * 100, 2),
    "%"
)

print(
    "Testing:",
    round(test_data["High_Risk"].mean() * 100, 2),
    "%"
)

Training rows: 1489038
Validation rows: 246026
Testing rows: 204135

Training period: 2010-03-16 00:00:00 to 2022-12-23 00:00:00
Validation period: 2023-01-03 00:00:00 to 2024-12-20 00:00:00
Testing period: 2025-01-02 00:00:00 to 2026-08-21 00:00:00

High-risk percentage by partition:
Training: 15.54 %
Validation: 16.01 %
Testing: 19.2 %


In [ ]:
model_data.to_parquet(
    "/content/drive/MyDrive/portfolio_risk_model_data.parquet",
    index=False
)

print("Processed modelling dataset saved.")

Processed modelling dataset saved.


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

baseline_predictions = np.zeros(
    len(validation_data),
    dtype=int
)

baseline_accuracy = accuracy_score(
    validation_data["High_Risk"],
    baseline_predictions
)

baseline_precision = precision_score(
    validation_data["High_Risk"],
    baseline_predictions,
    zero_division=0
)

baseline_recall = recall_score(
    validation_data["High_Risk"],
    baseline_predictions,
    zero_division=0
)

baseline_f1 = f1_score(
    validation_data["High_Risk"],
    baseline_predictions,
    zero_division=0
)

print("Baseline validation results")
print("Accuracy:", round(baseline_accuracy, 4))
print("Precision:", round(baseline_precision, 4))
print("Recall:", round(baseline_recall, 4))
print("F1 score:", round(baseline_f1, 4))

print("\nConfusion matrix:")
print(
    confusion_matrix(
        validation_data["High_Risk"],
        baseline_predictions
    )
)

Baseline validation results
Accuracy: 0.8399
Precision: 0.0
Recall: 0.0
F1 score: 0.0

Confusion matrix:
[[206631      0]
 [ 39395      0]]


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report
)

numeric_features = feature_columns
categorical_features = ["Sector"]

model_features = (
    numeric_features + categorical_features
)

X_train = train_data[model_features]
y_train = train_data["High_Risk"]

X_validation = validation_data[model_features]
y_validation = validation_data["High_Risk"]

# Standardize numerical variables and encode sectors
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),
        (
            "sector",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)

logistic_model = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        (
            "model",
            SGDClassifier(
                loss="log_loss",
                penalty="l2",
                class_weight="balanced",
                max_iter=2000,
                tol=1e-4,
                random_state=42
            )
        )
    ]
)

print("Training Logistic Regression...")

logistic_model.fit(
    X_train,
    y_train
)

print("Training completed.")

Training Logistic Regression...
Training completed.


In [ ]:
validation_probability = (
    logistic_model.predict_proba(
        X_validation
    )[:, 1]
)

validation_prediction = (
    validation_probability >= 0.50
).astype(int)

logistic_accuracy = accuracy_score(
    y_validation,
    validation_prediction
)

logistic_precision = precision_score(
    y_validation,
    validation_prediction
)

logistic_recall = recall_score(
    y_validation,
    validation_prediction
)

logistic_f1 = f1_score(
    y_validation,
    validation_prediction
)

logistic_roc_auc = roc_auc_score(
    y_validation,
    validation_probability
)

logistic_pr_auc = average_precision_score(
    y_validation,
    validation_probability
)

print("Logistic Regression validation results")
print("Accuracy:", round(logistic_accuracy, 4))
print("Precision:", round(logistic_precision, 4))
print("Recall:", round(logistic_recall, 4))
print("F1 score:", round(logistic_f1, 4))
print("ROC-AUC:", round(logistic_roc_auc, 4))
print("PR-AUC:", round(logistic_pr_auc, 4))

print("\nConfusion matrix:")
print(
    confusion_matrix(
        y_validation,
        validation_prediction
    )
)

print("\nClassification report:")
print(
    classification_report(
        y_validation,
        validation_prediction,
        digits=4
    )
)

Logistic Regression validation results
Accuracy: 0.4359
Precision: 0.1803
Recall: 0.7113
F1 score: 0.2877
ROC-AUC: 0.5769
PR-AUC: 0.203

Confusion matrix:
[[ 79219 127412]
 [ 11372  28023]]

Classification report:
              precision    recall  f1-score   support

           0     0.8745    0.3834    0.5331    206631
           1     0.1803    0.7113    0.2877     39395

    accuracy                         0.4359    246026
   macro avg     0.5274    0.5474    0.4104    246026
weighted avg     0.7633    0.4359    0.4938    246026



In [ ]:
def evaluate_risk_model(
    model_name,
    model,
    X,
    y,
    threshold=0.50
):
    probability = model.predict_proba(X)[:, 1]

    prediction = (
        probability >= threshold
    ).astype(int)

    results = {
        "Model": model_name,
        "Threshold": threshold,
        "Accuracy": accuracy_score(
            y, prediction
        ),
        "Precision": precision_score(
            y, prediction,
            zero_division=0
        ),
        "Recall": recall_score(
            y, prediction,
            zero_division=0
        ),
        "F1Score": f1_score(
            y, prediction,
            zero_division=0
        ),
        "ROCAUC": roc_auc_score(
            y, probability
        ),
        "PRAUC": average_precision_score(
            y, probability
        )
    }

    print(model_name)

    for metric, value in results.items():
        if metric != "Model":
            print(
                metric + ":",
                round(value, 4)
            )

    print("\nConfusion matrix:")
    print(confusion_matrix(y, prediction))

    return results, probability, prediction

In [ ]:
from sklearn.tree import DecisionTreeClassifier

decision_tree_model = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        (
            "model",
            DecisionTreeClassifier(
                max_depth=10,
                min_samples_leaf=500,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

print("Training Decision Tree...")

decision_tree_model.fit(
    X_train,
    y_train
)

print("Training completed.")

Training Decision Tree...
Training completed.


In [ ]:
decision_tree_results, (
    decision_tree_probability
), decision_tree_prediction = evaluate_risk_model(
    model_name="Decision Tree",
    model=decision_tree_model,
    X=X_validation,
    y=y_validation,
    threshold=0.50
)

Decision Tree
Threshold: 0.5
Accuracy: 0.5073
Precision: 0.1567
Recall: 0.4739
F1Score: 0.2355
ROCAUC: 0.4889
PRAUC: 0.1558

Confusion matrix:
[[106130 100501]
 [ 20726  18669]]


In [ ]:
!pip install xgboost -q

In [ ]:
from xgboost import XGBClassifier

xgb_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            "passthrough",
            numeric_features
        ),
        (
            "sector",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ],
    sparse_threshold=1.0
)

print("Preparing training data...")

X_train_xgb = xgb_preprocessor.fit_transform(
    X_train
)

X_validation_xgb = xgb_preprocessor.transform(
    X_validation
)

print("Training matrix shape:", X_train_xgb.shape)
print("Validation matrix shape:", X_validation_xgb.shape)

Preparing training data...
Training matrix shape: (1489038, 31)
Validation matrix shape: (246026, 31)


In [ ]:
negative_cases = (y_train == 0).sum()
positive_cases = (y_train == 1).sum()

risk_class_weight = (
    negative_cases / positive_cases
)

print(
    "High-risk class weight:",
    round(risk_class_weight, 2)
)

High-risk class weight: 5.43


In [ ]:
xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    min_child_weight=50,
    subsample=0.80,
    colsample_bytree=0.80,
    scale_pos_weight=risk_class_weight,
    reg_lambda=2,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    early_stopping_rounds=30,
    random_state=42,
    n_jobs=-1
)

print("Training XGBoost...")

xgb_model.fit(
    X_train_xgb,
    y_train,
    eval_set=[
        (
            X_validation_xgb,
            y_validation
        )
    ],
    verbose=25
)

print("XGBoost training completed.")

Training XGBoost...
[0]	validation_0-aucpr:0.15846
[25]	validation_0-aucpr:0.16356
[50]	validation_0-aucpr:0.16700
[75]	validation_0-aucpr:0.16546
[88]	validation_0-aucpr:0.16409
XGBoost training completed.


In [ ]:
xgb_results, (
    xgb_validation_probability
), xgb_validation_prediction = evaluate_risk_model(
    model_name="XGBoost",
    model=xgb_model,
    X=X_validation_xgb,
    y=y_validation,
    threshold=0.50
)

XGBoost
Threshold: 0.5
Accuracy: 0.6213
Precision: 0.1667
Recall: 0.3413
F1Score: 0.224
ROCAUC: 0.5142
PRAUC: 0.1674

Confusion matrix:
[[139412  67219]
 [ 25948  13447]]


In [ ]:
# Ensure correct chronological ordering
market_data = market_data.sort_values(
    ["Ticker", "Date"]
).reset_index(drop=True)

price_groups = market_data.groupby("Ticker")[
    "Adj Close"
]

# Collect adjusted closing prices for each of the next five days
future_prices = pd.concat(
    [
        price_groups.shift(-day)
        for day in range(1, 6)
    ],
    axis=1
)

future_prices.columns = [
    "Future_Day_1",
    "Future_Day_2",
    "Future_Day_3",
    "Future_Day_4",
    "Future_Day_5"
]

# Require all five future trading days
complete_future_window = (
    future_prices.notna().sum(axis=1) == 5
)

# Lowest price reached during the next five trading days
market_data["Future_Min_Price_5D"] = (
    future_prices.min(axis=1)
)

# Maximum decline from today's price
market_data["Future_Max_Drawdown_5D"] = (
    market_data["Future_Min_Price_5D"] /
    market_data["Adj Close"] - 1
)

# Corrected risk target
market_data["High_Risk"] = np.where(
    complete_future_window,
    (
        market_data[
            "Future_Max_Drawdown_5D"
        ] <= -0.03
    ).astype(int),
    np.nan
)

corrected_distribution = (
    market_data["High_Risk"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

print("Corrected target distribution:")
print(corrected_distribution)

Corrected target distribution:
High_Risk
0.0    76.69
1.0    23.31
Name: proportion, dtype: float64


In [ ]:
corrected_model_columns = (
    ["Ticker", "Company", "Sector", "Date"]
    + feature_columns
    + [
        "Future_Max_Drawdown_5D",
        "High_Risk"
    ]
)

model_data_v2 = market_data[
    corrected_model_columns
].copy()

model_data_v2 = model_data_v2.replace(
    [np.inf, -np.inf],
    np.nan
)

model_data_v2 = model_data_v2.dropna(
    subset=(
        feature_columns
        + [
            "Future_Max_Drawdown_5D",
            "High_Risk"
        ]
    )
).copy()

model_data_v2["High_Risk"] = (
    model_data_v2["High_Risk"].astype(int)
)

print("Corrected modelling rows:", len(model_data_v2))
print(
    "Overall high-risk rate:",
    round(
        model_data_v2["High_Risk"].mean() * 100,
        2
    ),
    "%"
)

Corrected modelling rows: 1944165
Overall high-risk rate: 23.3 %


In [ ]:
train_data_v2 = model_data_v2[
    model_data_v2["Date"] <= "2022-12-23"
].copy()

validation_data_v2 = model_data_v2[
    (model_data_v2["Date"] >= "2023-01-03") &
    (model_data_v2["Date"] <= "2024-12-20")
].copy()

test_data_v2 = model_data_v2[
    model_data_v2["Date"] >= "2025-01-02"
].copy()

print("Training rows:", len(train_data_v2))
print("Validation rows:", len(validation_data_v2))
print("Testing rows:", len(test_data_v2))

print("\nHigh-risk rate:")

print(
    "Training:",
    round(
        train_data_v2["High_Risk"].mean() * 100,
        2
    ),
    "%"
)

print(
    "Validation:",
    round(
        validation_data_v2["High_Risk"].mean() * 100,
        2
    ),
    "%"
)

print(
    "Testing:",
    round(
        test_data_v2["High_Risk"].mean() * 100,
        2
    ),
    "%"
)

Training rows: 1489038
Validation rows: 246026
Testing rows: 204135

High-risk rate:
Training: 22.74 %
Validation: 22.72 %
Testing: 28.25 %


In [ ]:
model_data_v2.to_parquet(
    "/content/drive/MyDrive/portfolio_risk_model_data_v2.parquet",
    index=False
)

print("Corrected dataset saved.")

Corrected dataset saved.


In [ ]:
X_train_v2 = train_data_v2[model_features]
y_train_v2 = train_data_v2["High_Risk"]

X_validation_v2 = validation_data_v2[
    model_features
]

y_validation_v2 = validation_data_v2[
    "High_Risk"
]

logistic_model_v2 = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        (
            "model",
            SGDClassifier(
                loss="log_loss",
                penalty="l2",
                class_weight="balanced",
                max_iter=2000,
                tol=1e-4,
                random_state=42
            )
        )
    ]
)

print("Training corrected Logistic Regression...")

logistic_model_v2.fit(
    X_train_v2,
    y_train_v2
)

print("Training completed.")

Training corrected Logistic Regression...
Training completed.


In [ ]:
logistic_v2_results, (
    logistic_v2_probability
), logistic_v2_prediction = evaluate_risk_model(
    model_name="Corrected Logistic Regression",
    model=logistic_model_v2,
    X=X_validation_v2,
    y=y_validation_v2,
    threshold=0.50
)

Corrected Logistic Regression
Threshold: 0.5
Accuracy: 0.4863
Precision: 0.2618
Recall: 0.6931
F1Score: 0.3801
ROCAUC: 0.5932
PRAUC: 0.2994

Confusion matrix:
[[ 80907 109220]
 [ 17157  38742]]


In [ ]:
X_train_xgb_v2 = (
    xgb_preprocessor.fit_transform(
        X_train_v2
    )
)

X_validation_xgb_v2 = (
    xgb_preprocessor.transform(
        X_validation_v2
    )
)

negative_v2 = (y_train_v2 == 0).sum()
positive_v2 = (y_train_v2 == 1).sum()

risk_class_weight_v2 = (
    negative_v2 / positive_v2
)

print(
    "Corrected class weight:",
    round(risk_class_weight_v2, 2)
)

print(
    "Training matrix:",
    X_train_xgb_v2.shape
)

print(
    "Validation matrix:",
    X_validation_xgb_v2.shape
)

Corrected class weight: 3.4
Training matrix: (1489038, 31)
Validation matrix: (246026, 31)


In [ ]:
xgb_model_v2 = XGBClassifier(
    n_estimators=700,
    max_depth=6,
    learning_rate=0.04,
    min_child_weight=50,
    subsample=0.80,
    colsample_bytree=0.80,
    scale_pos_weight=risk_class_weight_v2,
    reg_lambda=3,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    early_stopping_rounds=40,
    random_state=42,
    n_jobs=-1
)

print("Training corrected XGBoost...")

xgb_model_v2.fit(
    X_train_xgb_v2,
    y_train_v2,
    eval_set=[
        (
            X_validation_xgb_v2,
            y_validation_v2
        )
    ],
    verbose=50
)

print("Training completed.")

Training corrected XGBoost...
[0]	validation_0-aucpr:0.25566
[48]	validation_0-aucpr:0.27980
Training completed.


In [ ]:
xgb_v2_results, (
    xgb_v2_probability
), xgb_v2_prediction = evaluate_risk_model(
    model_name="Corrected XGBoost",
    model=xgb_model_v2,
    X=X_validation_xgb_v2,
    y=y_validation_v2,
    threshold=0.50
)

Corrected XGBoost
Threshold: 0.5
Accuracy: 0.5504
Precision: 0.2653
Recall: 0.5531
F1Score: 0.3586
ROCAUC: 0.5738
PRAUC: 0.2858

Confusion matrix:
[[104488  85639]
 [ 24982  30917]]


In [ ]:
from sklearn.metrics import fbeta_score

threshold_results = []

for threshold in np.arange(
    0.30,
    0.76,
    0.025
):
    prediction = (
        logistic_v2_probability >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_validation_v2,
        prediction
    ).ravel()

    threshold_results.append(
        {
            "Threshold": threshold,
            "Accuracy": accuracy_score(
                y_validation_v2,
                prediction
            ),
            "Precision": precision_score(
                y_validation_v2,
                prediction,
                zero_division=0
            ),
            "Recall": recall_score(
                y_validation_v2,
                prediction,
                zero_division=0
            ),
            "F1Score": f1_score(
                y_validation_v2,
                prediction,
                zero_division=0
            ),
            "F2Score": fbeta_score(
                y_validation_v2,
                prediction,
                beta=2,
                zero_division=0
            ),
            "Alert_Rate": prediction.mean(),
            "False_Positives": fp,
            "False_Negatives": fn
        }
    )

threshold_table = pd.DataFrame(
    threshold_results
)

threshold_table[
    [
        "Threshold",
        "Precision",
        "Recall",
        "F1Score",
        "F2Score",
        "Alert_Rate",
        "False_Positives",
        "False_Negatives"
    ]
].round(4)

,Threshold,Precision,Recall,F1Score,F2Score,Alert_Rate,False_Positives,False_Negatives
0,0.300,0.2272,0.9999,0.3703,0.5952,0.9998,190091,3
1,0.325,0.2275,0.9980,0.3706,0.5950,0.9967,189426,110
2,0.350,0.2287,0.9894,0.3715,0.5941,0.9832,186574,591
3,0.375,0.2308,0.9734,0.3732,0.5923,0.9581,181298,1486
4,0.400,0.2333,0.9515,0.3748,0.5890,0.9265,174758,2710
5,0.425,0.2372,0.9197,0.3772,0.5838,0.8808,165299,4490
6,0.450,0.2435,0.8690,0.3804,0.5741,0.8109,150931,7322
7,0.475,0.2520,0.7917,0.3823,0.5543,0.7137,131346,11646
8,0.500,0.2618,0.6931,0.3801,0.5213,0.6014,109220,17157
9,0.525,0.2745,0.5882,0.3744,0.4788,0.4868,86889,23017


In [ ]:
best_f1_threshold = threshold_table.loc[
    threshold_table["F1Score"].idxmax()
]

recall_candidates = threshold_table[
    threshold_table["Recall"] >= 0.60
]

best_operational_threshold = (
    recall_candidates
    .sort_values(
        ["Precision", "F1Score"],
        ascending=False
    )
    .iloc[0]
)

print("Best F1 threshold:")
print(best_f1_threshold.round(4))

print("\nBest threshold with at least 60% recall:")
print(best_operational_threshold.round(4))

Best F1 threshold:
Threshold               0.4750
Accuracy                0.4188
Precision               0.2520
Recall                  0.7917
F1Score                 0.3823
F2Score                 0.5543
Alert_Rate              0.7137
False_Positives    131346.0000
False_Negatives     11646.0000
Name: 7, dtype: float64

Best threshold with at least 60% recall:
Threshold               0.5000
Accuracy                0.4863
Precision               0.2618
Recall                  0.6931
F1Score                 0.3801
F2Score                 0.5213
Alert_Rate              0.6014
False_Positives    109220.0000
False_Negatives     17157.0000
Name: 8, dtype: float64


In [ ]:
# Ensure chronological ordering
market_data = market_data.sort_values(
    ["Ticker", "Date"]
).reset_index(drop=True)

return_groups = market_data.groupby("Ticker")[
    "Daily_Return"
]

# Obtain each of the next five daily returns
future_daily_returns = pd.concat(
    [
        return_groups.shift(-day)
        for day in range(1, 6)
    ],
    axis=1
)

future_daily_returns.columns = [
    "Future_Return_Day_1",
    "Future_Return_Day_2",
    "Future_Return_Day_3",
    "Future_Return_Day_4",
    "Future_Return_Day_5"
]

# Require all five future trading days
complete_volatility_window = (
    future_daily_returns.notna().sum(axis=1) == 5
)

# Annualized five-day realized volatility
market_data["Future_Volatility_5D"] = (
    np.sqrt(
        future_daily_returns.pow(2).mean(axis=1)
    )
    * np.sqrt(252)
)

market_data.loc[
    ~complete_volatility_window,
    "Future_Volatility_5D"
] = np.nan

print(
    market_data[
        [
            "Ticker",
            "Date",
            "Volatility_20D",
            "Future_Volatility_5D"
        ]
    ].tail(10)
)

        Ticker       Date  Volatility_20D  Future_Volatility_5D
1974096    ZTS 2026-08-14        0.023857              0.416001
1974097    ZTS 2026-08-17        0.023942              0.411642
1974098    ZTS 2026-08-18        0.024076              0.400367
1974099    ZTS 2026-08-19        0.025195              0.304258
1974100    ZTS 2026-08-20        0.025723              0.348383
1974101    ZTS 2026-08-21        0.026855                   NaN
1974102    ZTS 2026-08-24        0.026545                   NaN
1974103    ZTS 2026-08-25        0.026493                   NaN
1974104    ZTS 2026-08-26        0.026473                   NaN
1974105    ZTS 2026-08-27        0.026911                   NaN


In [ ]:
training_volatility = market_data[
    (
        market_data["Date"] <= "2022-12-23"
    )
    &
    (
        market_data[
            "Future_Volatility_5D"
        ].notna()
    )
].copy()

volatility_thresholds = (
    training_volatility
    .groupby("Ticker")[
        "Future_Volatility_5D"
    ]
    .quantile(0.75)
    .rename("Volatility_Threshold")
    .reset_index()
)

market_data = market_data.merge(
    volatility_thresholds,
    on="Ticker",
    how="left",
    validate="many_to_one"
)

market_data["High_Volatility_Risk"] = np.where(
    market_data["Future_Volatility_5D"].notna(),
    (
        market_data["Future_Volatility_5D"]
        >=
        market_data["Volatility_Threshold"]
    ).astype(int),
    np.nan
)

print(
    volatility_thresholds[
        "Volatility_Threshold"
    ].describe()
)

count    494.000000
mean       0.319791
std        0.111846
min        0.168430
25%        0.251985
50%        0.292682
75%        0.363217
max        0.983441
Name: Volatility_Threshold, dtype: float64


In [ ]:
for period_name, start_date, end_date in [
    (
        "Training",
        "2010-01-01",
        "2022-12-23"
    ),
    (
        "Validation",
        "2023-01-03",
        "2024-12-20"
    ),
    (
        "Testing",
        "2025-01-02",
        "2026-08-21"
    )
]:
    period = market_data[
        (
            market_data["Date"] >= start_date
        )
        &
        (
            market_data["Date"] <= end_date
        )
        &
        (
            market_data[
                "High_Volatility_Risk"
            ].notna()
        )
    ]

    risk_rate = (
        period[
            "High_Volatility_Risk"
        ].mean() * 100
    )

    print(
        period_name,
        "high-volatility rate:",
        round(risk_rate, 2),
        "%"
    )

Training high-volatility rate: 25.0 %
Validation high-volatility rate: 21.13 %
Testing high-volatility rate: 34.26 %


In [ ]:
 # Absolute daily movement
market_data["Absolute_Return"] = (
    market_data["Daily_Return"].abs()
)

ticker_groups = market_data.groupby("Ticker")

# Short-term volatility
market_data["Volatility_5D"] = ticker_groups[
    "Daily_Return"
].transform(
    lambda values: values.rolling(5).std()
)

# Average absolute movement
market_data["Mean_Absolute_Return_5D"] = (
    ticker_groups["Absolute_Return"]
    .transform(
        lambda values: values.rolling(5).mean()
    )
)

market_data["Mean_Absolute_Return_20D"] = (
    ticker_groups["Absolute_Return"]
    .transform(
        lambda values: values.rolling(20).mean()
    )
)

# Largest recent movement
market_data["Maximum_Absolute_Return_5D"] = (
    ticker_groups["Absolute_Return"]
    .transform(
        lambda values: values.rolling(5).max()
    )
)

market_data["Maximum_Absolute_Return_20D"] = (
    ticker_groups["Absolute_Return"]
    .transform(
        lambda values: values.rolling(20).max()
    )
)

# Average intraday trading ranges
market_data["Average_Range_5D"] = (
    ticker_groups["Daily_Price_Range"]
    .transform(
        lambda values: values.rolling(5).mean()
    )
)

market_data["Average_Range_20D"] = (
    ticker_groups["Daily_Price_Range"]
    .transform(
        lambda values: values.rolling(20).mean()
    )
)

# Volatility acceleration
market_data["Volatility_Acceleration"] = (
    market_data["Volatility_5D"] /
    market_data["Volatility_20D"]
)

market_data["Range_Acceleration"] = (
    market_data["Average_Range_5D"] /
    market_data["Average_Range_20D"]
)

print("Volatility-focused features created.")

Volatility-focused features created.


In [ ]:
volatility_features = feature_columns + [
    "Volatility_5D",
    "Mean_Absolute_Return_5D",
    "Mean_Absolute_Return_20D",
    "Maximum_Absolute_Return_5D",
    "Maximum_Absolute_Return_20D",
    "Average_Range_5D",
    "Average_Range_20D",
    "Volatility_Acceleration",
    "Range_Acceleration"
]

volatility_model_columns = (
    [
        "Ticker",
        "Company",
        "Sector",
        "Date"
    ]
    + volatility_features
    + [
        "Future_Volatility_5D",
        "Volatility_Threshold",
        "High_Volatility_Risk"
    ]
)

volatility_model_data = market_data[
    volatility_model_columns
].copy()

volatility_model_data = (
    volatility_model_data
    .replace([np.inf, -np.inf], np.nan)
    .dropna(
        subset=(
            volatility_features
            + ["High_Volatility_Risk"]
        )
    )
    .copy()
)

volatility_model_data[
    "High_Volatility_Risk"
] = volatility_model_data[
    "High_Volatility_Risk"
].astype(int)

print(
    "Volatility modelling rows:",
    len(volatility_model_data)
)

print(
    "Available securities:",
    volatility_model_data["Ticker"].nunique()
)

Volatility modelling rows: 1941226
Available securities: 502


In [ ]:
vol_train = volatility_model_data[
    volatility_model_data["Date"] <= "2022-12-23"
].copy()

vol_validation = volatility_model_data[
    (
        volatility_model_data["Date"]
        >= "2023-01-03"
    )
    &
    (
        volatility_model_data["Date"]
        <= "2024-12-20"
    )
].copy()

vol_test = volatility_model_data[
    volatility_model_data["Date"] >= "2025-01-02"
].copy()

print("Training rows:", len(vol_train))
print("Validation rows:", len(vol_validation))
print("Testing rows:", len(vol_test))

print(
    "Training risk rate:",
    round(
        vol_train[
            "High_Volatility_Risk"
        ].mean() * 100,
        2
    )
)

print(
    "Validation risk rate:",
    round(
        vol_validation[
            "High_Volatility_Risk"
        ].mean() * 100,
        2
    )
)

print(
    "Testing risk rate:",
    round(
        vol_test[
            "High_Volatility_Risk"
        ].mean() * 100,
        2
    )
)

Training rows: 1486178
Validation rows: 245947
Testing rows: 204135
Training risk rate: 25.01
Validation risk rate: 21.13
Testing risk rate: 34.26


In [ ]:
volatility_model_data.to_parquet(
    "/content/drive/MyDrive/volatility_risk_model_data.parquet",
    index=False
)

print("Volatility dataset saved.")

Volatility dataset saved.


In [ ]:
volatility_categorical_features = [
    "Ticker",
    "Sector"
]

volatility_model_features = (
    volatility_features
    + volatility_categorical_features
)

X_vol_train = vol_train[
    volatility_model_features
]

y_vol_train = vol_train[
    "High_Volatility_Risk"
]

X_vol_validation = vol_validation[
    volatility_model_features
]

y_vol_validation = vol_validation[
    "High_Volatility_Risk"
]

volatility_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            volatility_features
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            volatility_categorical_features
        )
    ]
)

In [ ]:
volatility_logistic_model = Pipeline(
    steps=[
        (
            "preprocessing",
            volatility_preprocessor
        ),
        (
            "model",
            SGDClassifier(
                loss="log_loss",
                penalty="l2",
                class_weight="balanced",
                max_iter=2000,
                tol=1e-4,
                random_state=42
            )
        )
    ]
)

print(
    "Training Volatility Logistic Regression..."
)

volatility_logistic_model.fit(
    X_vol_train,
    y_vol_train
)

print("Training completed.")

Training Volatility Logistic Regression...
Training completed.


In [ ]:
vol_logistic_results, (
    vol_logistic_probability
), vol_logistic_prediction = evaluate_risk_model(
    model_name="Volatility Logistic Regression",
    model=volatility_logistic_model,
    X=X_vol_validation,
    y=y_vol_validation,
    threshold=0.50
)

Volatility Logistic Regression
Threshold: 0.5
Accuracy: 0.7142
Precision: 0.3498
Recall: 0.4105
F1Score: 0.3777
ROCAUC: 0.667
PRAUC: 0.3431

Confusion matrix:
[[154319  39658]
 [ 30635  21335]]


In [ ]:
vol_xgb_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            "passthrough",
            volatility_features
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            volatility_categorical_features
        )
    ],
    sparse_threshold=1.0
)

print("Preparing XGBoost matrices...")

X_vol_train_xgb = (
    vol_xgb_preprocessor.fit_transform(
        X_vol_train
    )
)

X_vol_validation_xgb = (
    vol_xgb_preprocessor.transform(
        X_vol_validation
    )
)

negative_volatility = (
    y_vol_train == 0
).sum()

positive_volatility = (
    y_vol_train == 1
).sum()

volatility_class_weight = (
    negative_volatility /
    positive_volatility
)

print(
    "Training matrix:",
    X_vol_train_xgb.shape
)

print(
    "Validation matrix:",
    X_vol_validation_xgb.shape
)

print(
    "Class weight:",
    round(volatility_class_weight, 2)
)

Preparing XGBoost matrices...
Training matrix: (1486178, 533)
Validation matrix: (245947, 533)
Class weight: 3.0


In [ ]:
volatility_xgb_model = XGBClassifier(
    n_estimators=800,
    max_depth=7,
    learning_rate=0.04,
    min_child_weight=50,
    subsample=0.85,
    colsample_bytree=0.85,
    scale_pos_weight=volatility_class_weight,
    reg_lambda=3,
    reg_alpha=0.10,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    early_stopping_rounds=50,
    random_state=42,
    n_jobs=-1
)

print("Training Volatility XGBoost...")

volatility_xgb_model.fit(
    X_vol_train_xgb,
    y_vol_train,
    eval_set=[
        (
            X_vol_validation_xgb,
            y_vol_validation
        )
    ],
    verbose=50
)

print("Training completed.")

Training Volatility XGBoost...
[0]	validation_0-aucpr:0.24828
[50]	validation_0-aucpr:0.29476
[100]	validation_0-aucpr:0.30606
[150]	validation_0-aucpr:0.31212
[200]	validation_0-aucpr:0.31686
[250]	validation_0-aucpr:0.31867
[300]	validation_0-aucpr:0.32141
[350]	validation_0-aucpr:0.32260
[400]	validation_0-aucpr:0.32374
[450]	validation_0-aucpr:0.32449
[500]	validation_0-aucpr:0.32549
[550]	validation_0-aucpr:0.32618
[600]	validation_0-aucpr:0.32696
[650]	validation_0-aucpr:0.32742
[700]	validation_0-aucpr:0.32851
[750]	validation_0-aucpr:0.32935
[799]	validation_0-aucpr:0.33015
Training completed.


In [ ]:
vol_xgb_results, (
    vol_xgb_probability
), vol_xgb_prediction = evaluate_risk_model(
    model_name="Volatility XGBoost",
    model=volatility_xgb_model,
    X=X_vol_validation_xgb,
    y=y_vol_validation,
    threshold=0.50
)

Volatility XGBoost
Threshold: 0.5
Accuracy: 0.6306
Precision: 0.3043
Recall: 0.5819
F1Score: 0.3997
ROCAUC: 0.6567
PRAUC: 0.3302

Confusion matrix:
[[124848  69129]
 [ 21727  30243]]


In [ ]:
from sklearn.metrics import fbeta_score

def create_threshold_table(
    model_name,
    actual,
    probability
):
    records = []

    for threshold in np.arange(
        0.20,
        0.81,
        0.025
    ):
        prediction = (
            probability >= threshold
        ).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            actual,
            prediction
        ).ravel()

        records.append(
            {
                "Model": model_name,
                "Threshold": threshold,
                "Precision": precision_score(
                    actual,
                    prediction,
                    zero_division=0
                ),
                "Recall": recall_score(
                    actual,
                    prediction,
                    zero_division=0
                ),
                "F1Score": f1_score(
                    actual,
                    prediction,
                    zero_division=0
                ),
                "F2Score": fbeta_score(
                    actual,
                    prediction,
                    beta=2,
                    zero_division=0
                ),
                "Alert_Rate": prediction.mean(),
                "False_Positives": fp,
                "False_Negatives": fn
            }
        )

    return pd.DataFrame(records)

In [ ]:
logistic_thresholds = create_threshold_table(
    model_name="Logistic Regression",
    actual=y_vol_validation,
    probability=vol_logistic_probability
)

xgb_thresholds = create_threshold_table(
    model_name="XGBoost",
    actual=y_vol_validation,
    probability=vol_xgb_probability
)

combined_thresholds = pd.concat(
    [
        logistic_thresholds,
        xgb_thresholds
    ],
    ignore_index=True
)

print(
    "Threshold table created:",
    combined_thresholds.shape
)

Threshold table created: (50, 9)


In [ ]:
best_f1_results = (
    combined_thresholds
    .sort_values(
        "F1Score",
        ascending=False
    )
    .groupby("Model")
    .head(1)
)

best_f1_results[
    [
        "Model",
        "Threshold",
        "Precision",
        "Recall",
        "F1Score",
        "F2Score",
        "Alert_Rate",
        "False_Positives",
        "False_Negatives"
    ]
].round(4)

,Model,Threshold,Precision,Recall,F1Score,F2Score,Alert_Rate,False_Positives,False_Negatives
8,Logistic Regression,0.400,0.2964,0.6552,0.4082,0.5275,0.4671,80825,17920
36,XGBoost,0.475,0.2910,0.6454,0.4012,0.5190,0.4686,81699,18430


In [ ]:
import gc

variables_to_remove = [
    "X_vol_validation_xgb",
    "vol_xgb_probability",
    "vol_xgb_prediction"
]

for variable in variables_to_remove:
    if variable in globals():
        del globals()[variable]

gc.collect()

print("Unused XGBoost objects removed.")

Unused XGBoost objects removed.


In [ ]:
X_vol_test = vol_test[
    volatility_model_features
]

y_vol_test = vol_test[
    "High_Volatility_Risk"
]

test_probability = (
    volatility_logistic_model.predict_proba(
        X_vol_test
    )[:, 1]
)

selected_threshold = 0.40

test_prediction = (
    test_probability >= selected_threshold
).astype(int)

In [ ]:
test_accuracy = accuracy_score(
    y_vol_test,
    test_prediction
)

test_precision = precision_score(
    y_vol_test,
    test_prediction,
    zero_division=0
)

test_recall = recall_score(
    y_vol_test,
    test_prediction,
    zero_division=0
)

test_f1 = f1_score(
    y_vol_test,
    test_prediction,
    zero_division=0
)

test_roc_auc = roc_auc_score(
    y_vol_test,
    test_probability
)

test_pr_auc = average_precision_score(
    y_vol_test,
    test_probability
)

test_confusion_matrix = confusion_matrix(
    y_vol_test,
    test_prediction
)

print("Final test results")
print("Threshold:", selected_threshold)
print("Accuracy:", round(test_accuracy, 4))
print("Precision:", round(test_precision, 4))
print("Recall:", round(test_recall, 4))
print("F1 score:", round(test_f1, 4))
print("ROC-AUC:", round(test_roc_auc, 4))
print("PR-AUC:", round(test_pr_auc, 4))

print("\nConfusion matrix:")
print(test_confusion_matrix)

Final test results
Threshold: 0.4
Accuracy: 0.5083
Precision: 0.4014
Recall: 0.8858
F1 score: 0.5524
ROC-AUC: 0.6871
PR-AUC: 0.525

Confusion matrix:
[[41815 92386]
 [ 7987 61947]]


In [ ]:
import joblib

joblib.dump(
    volatility_logistic_model,
    "/content/drive/MyDrive/volatility_logistic_model.joblib"
)

print("Selected model saved.")

Selected model saved.


In [ ]:
powerbi_columns = [
    "Ticker",
    "Company",
    "Sector",
    "Date",
    "Return_5D",
    "Return_20D",
    "Volatility_5D",
    "Volatility_20D",
    "Volatility_Acceleration",
    "Drawdown_20D",
    "Volume_Ratio_20D",
    "SPY_Return_5D",
    "VIX",
    "Treasury_Yield",
    "Future_Volatility_5D",
    "Volatility_Threshold",
    "High_Volatility_Risk"
]

scored_test_data = vol_test[
    powerbi_columns
].copy()

# This is called a risk score rather than a calibrated probability
scored_test_data["Risk_Score"] = (
    test_probability
)

scored_test_data[
    "Predicted_High_Volatility"
] = test_prediction

scored_test_data["Risk_Level"] = pd.cut(
    scored_test_data["Risk_Score"],
    bins=[
        -np.inf,
        0.40,
        0.60,
        0.75,
        np.inf
    ],
    labels=[
        "Low",
        "Elevated",
        "High",
        "Critical"
    ],
    right=False
)

scored_test_data["Prediction_Outcome"] = np.select(
    [
        (
            scored_test_data["High_Volatility_Risk"] == 1
        )
        &
        (
            scored_test_data[
                "Predicted_High_Volatility"
            ] == 1
        ),
        (
            scored_test_data["High_Volatility_Risk"] == 0
        )
        &
        (
            scored_test_data[
                "Predicted_High_Volatility"
            ] == 0
        ),
        (
            scored_test_data["High_Volatility_Risk"] == 0
        )
        &
        (
            scored_test_data[
                "Predicted_High_Volatility"
            ] == 1
        ),
        (
            scored_test_data["High_Volatility_Risk"] == 1
        )
        &
        (
            scored_test_data[
                "Predicted_High_Volatility"
            ] == 0
        )
    ],
    [
        "True Positive",
        "True Negative",
        "False Positive",
        "False Negative"
    ],
    default="Unknown"
)

scored_test_data.head()

,Ticker,Company,Sector,Date,Return_5D,Return_20D,Volatility_5D,Volatility_20D,Volatility_Acceleration,Drawdown_20D,...,SPY_Return_5D,VIX,Treasury_Yield,Future_Volatility_5D,Volatility_Threshold,High_Volatility_Risk,Risk_Score,Predicted_High_Volatility,Risk_Level,Prediction_Outcome
3774,A,Agilent Technologies,Health Care,2025-01-02,-0.017806,-0.040231,0.003877,0.012135,0.319472,-0.071690,...,-0.027707,17.930000,4.575,0.140002,0.308777,0,0.437659,1,Elevated,False Positive
3775,A,Agilent Technologies,Health Care,2025-01-03,0.000816,-0.030379,0.010087,0.012725,0.792649,-0.055967,...,-0.015615,16.129999,4.596,0.242214,0.308777,0,0.394849,0,Low,True Negative
3776,A,Agilent Technologies,Health Care,2025-01-06,0.008427,-0.008616,0.010217,0.012292,0.831212,-0.050818,...,0.000588,16.040001,4.618,0.250295,0.308777,0,0.373538,0,Low,True Negative
3777,A,Agilent Technologies,Health Care,2025-01-07,0.024133,-0.020743,0.008660,0.011515,0.752085,-0.044000,...,0.000697,17.820000,4.683,0.252140,0.308777,0,0.446784,1,Elevated,False Positive
3778,A,Agilent Technologies,Health Care,2025-01-08,0.019801,-0.046852,0.009280,0.009854,0.941710,-0.032951,...,0.005818,17.700001,4.693,0.280309,0.308777,0,0.433930,1,Elevated,False Positive


In [ ]:
model_performance = pd.DataFrame(
    [
        {
            "Model": "Logistic Regression",
            "Dataset": "Validation",
            "Threshold": 0.40,
            "Accuracy": 0.7142,
            "Precision": 0.3498,
            "Recall": 0.4105,
            "F1Score": 0.3777,
            "ROCAUC": 0.6670,
            "PRAUC": 0.3431
        },
        {
            "Model": "XGBoost",
            "Dataset": "Validation",
            "Threshold": 0.475,
            "Accuracy": 0.6306,
            "Precision": 0.3043,
            "Recall": 0.5819,
            "F1Score": 0.3997,
            "ROCAUC": 0.6567,
            "PRAUC": 0.3302
        },
        {
            "Model": "Selected Logistic Regression",
            "Dataset": "Final Test",
            "Threshold": 0.40,
            "Accuracy": 0.5083,
            "Precision": 0.4014,
            "Recall": 0.8858,
            "F1Score": 0.5524,
            "ROCAUC": 0.6871,
            "PRAUC": 0.5250
        }
    ]
)

model_performance

,Model,Dataset,Threshold,Accuracy,Precision,Recall,F1Score,ROCAUC,PRAUC
0,Logistic Regression,Validation,0.400,0.7142,0.3498,0.4105,0.3777,0.6670,0.3431
1,XGBoost,Validation,0.475,0.6306,0.3043,0.5819,0.3997,0.6567,0.3302
2,Selected Logistic Regression,Final Test,0.400,0.5083,0.4014,0.8858,0.5524,0.6871,0.5250


In [ ]:
import gc

print("Recreating the XGBoost validation matrix...")

X_vol_validation_xgb_temp = (
    vol_xgb_preprocessor.transform(
        X_vol_validation
    )
)

vol_xgb_probability = (
    volatility_xgb_model.predict_proba(
        X_vol_validation_xgb_temp
    )[:, 1]
)

# Remove the temporary matrix after obtaining probabilities
del X_vol_validation_xgb_temp
gc.collect()

print(
    "XGBoost probabilities restored:",
    len(vol_xgb_probability)
)

Recreating the XGBoost validation matrix...
XGBoost probabilities restored: 245947


In [ ]:
logistic_validation_selected = (
    vol_logistic_probability >= 0.40
).astype(int)

xgb_validation_selected = (
    vol_xgb_probability >= 0.475
).astype(int)

In [ ]:
# Predictions using the selected validation thresholds
logistic_validation_selected = (
    vol_logistic_probability >= 0.40
).astype(int)

xgb_validation_selected = (
    vol_xgb_probability >= 0.475
).astype(int)


def performance_row(
    model_name,
    dataset_name,
    threshold,
    actual,
    probability,
    prediction
):
    return {
        "Model": model_name,
        "Dataset": dataset_name,
        "Threshold": threshold,
        "Accuracy": accuracy_score(
            actual,
            prediction
        ),
        "Precision": precision_score(
            actual,
            prediction,
            zero_division=0
        ),
        "Recall": recall_score(
            actual,
            prediction,
            zero_division=0
        ),
        "F1Score": f1_score(
            actual,
            prediction,
            zero_division=0
        ),
        "ROCAUC": roc_auc_score(
            actual,
            probability
        ),
        "PRAUC": average_precision_score(
            actual,
            probability
        ),
        "AlertRate": prediction.mean()
    }


model_performance = pd.DataFrame(
    [
        performance_row(
            "Logistic Regression",
            "Validation",
            0.40,
            y_vol_validation,
            vol_logistic_probability,
            logistic_validation_selected
        ),
        performance_row(
            "XGBoost",
            "Validation",
            0.475,
            y_vol_validation,
            vol_xgb_probability,
            xgb_validation_selected
        ),
        performance_row(
            "Selected Logistic Regression",
            "Final Test",
            0.40,
            y_vol_test,
            test_probability,
            test_prediction
        )
    ]
)

model_performance.round(4)

,Model,Dataset,Threshold,Accuracy,Precision,Recall,F1Score,ROCAUC,PRAUC,AlertRate
0,Logistic Regression,Validation,0.400,0.5985,0.2964,0.6552,0.4082,0.6670,0.3431,0.4671
1,XGBoost,Validation,0.475,0.5929,0.2910,0.6454,0.4012,0.6567,0.3302,0.4686
2,Selected Logistic Regression,Final Test,0.400,0.5083,0.4014,0.8858,0.5524,0.6871,0.5250,0.7560


In [ ]:
scored_test_data[
    [
        "Ticker",
        "Date",
        "Risk_Score",
        "Risk_Level",
        "High_Volatility_Risk",
        "Predicted_High_Volatility",
        "Prediction_Outcome"
    ]
].head(10)

,Ticker,Date,Risk_Score,Risk_Level,High_Volatility_Risk,Predicted_High_Volatility,Prediction_Outcome
3774,A,2025-01-02,0.437659,Elevated,0,1,False Positive
3775,A,2025-01-03,0.394849,Low,0,0,True Negative
3776,A,2025-01-06,0.373538,Low,0,0,True Negative
3777,A,2025-01-07,0.446784,Elevated,0,1,False Positive
3778,A,2025-01-08,0.433930,Elevated,0,1,False Positive
3779,A,2025-01-10,0.520054,Elevated,0,1,False Positive
3780,A,2025-01-13,0.534057,Elevated,1,1,True Positive
3781,A,2025-01-14,0.571246,Elevated,1,1,True Positive
3782,A,2025-01-15,0.513512,Elevated,0,1,False Positive
3783,A,2025-01-16,0.487261,Elevated,0,1,False Positive


In [ ]:
scored_test_data.to_csv(
    "/content/drive/MyDrive/portfolio_risk_scored_data.csv",
    index=False
)

model_performance.to_csv(
    "/content/drive/MyDrive/portfolio_risk_model_performance.csv",
    index=False
)

print("Power BI files exported successfully.")
print("Scored rows:", len(scored_test_data))
print("Performance rows:", len(model_performance))

Power BI files exported successfully.
Scored rows: 204135
Performance rows: 3


In [ ]:
# Obtain transformed feature names and model coefficients
transformed_names = (
    volatility_logistic_model
    .named_steps["preprocessing"]
    .get_feature_names_out()
)

coefficients = (
    volatility_logistic_model
    .named_steps["model"]
    .coef_[0]
)

coefficient_table = pd.DataFrame(
    {
        "Feature": transformed_names,
        "Coefficient": coefficients
    }
)

# Retain analytical indicators rather than company identifiers
feature_importance = coefficient_table[
    coefficient_table["Feature"].str.startswith(
        "numeric__"
    )
].copy()

feature_importance["Feature"] = (
    feature_importance["Feature"]
    .str.replace(
        "numeric__",
        "",
        regex=False
    )
)

feature_importance["Absolute_Importance"] = (
    feature_importance["Coefficient"].abs()
)

feature_importance["Direction"] = np.where(
    feature_importance["Coefficient"] >= 0,
    "Increases Risk Score",
    "Decreases Risk Score"
)

feature_importance["Importance_Percentage"] = (
    feature_importance["Absolute_Importance"]
    /
    feature_importance[
        "Absolute_Importance"
    ].sum()
)

feature_importance = (
    feature_importance
    .sort_values(
        "Absolute_Importance",
        ascending=False
    )
    .reset_index(drop=True)
)

feature_importance.head(15)

,Feature,Coefficient,Absolute_Importance,Direction,Importance_Percentage
0,VIX,0.764398,0.764398,Increases Risk Score,0.226059
1,Average_Range_20D,0.481137,0.481137,Increases Risk Score,0.142289
2,Maximum_Absolute_Return_20D,-0.290810,0.290810,Decreases Risk Score,0.086003
3,Mean_Absolute_Return_20D,0.239341,0.239341,Increases Risk Score,0.070781
4,Volatility_20D,0.238120,0.238120,Increases Risk Score,0.070420
5,SPY_Volatility_20D,-0.197137,0.197137,Decreases Risk Score,0.058300
6,Volume_Ratio_20D,0.119815,0.119815,Increases Risk Score,0.035434
7,Average_Range_5D,0.102872,0.102872,Increases Risk Score,0.030423
8,Price_to_SMA10,-0.102680,0.102680,Decreases Risk Score,0.030366
9,Mean_Absolute_Return_5D,-0.092090,0.092090,Decreases Risk Score,0.027234


In [ ]:
feature_importance.head(15)

,Feature,Coefficient,Absolute_Importance,Direction,Importance_Percentage
0,VIX,0.764398,0.764398,Increases Risk Score,0.226059
1,Average_Range_20D,0.481137,0.481137,Increases Risk Score,0.142289
2,Maximum_Absolute_Return_20D,-0.290810,0.290810,Decreases Risk Score,0.086003
3,Mean_Absolute_Return_20D,0.239341,0.239341,Increases Risk Score,0.070781
4,Volatility_20D,0.238120,0.238120,Increases Risk Score,0.070420
5,SPY_Volatility_20D,-0.197137,0.197137,Decreases Risk Score,0.058300
6,Volume_Ratio_20D,0.119815,0.119815,Increases Risk Score,0.035434
7,Average_Range_5D,0.102872,0.102872,Increases Risk Score,0.030423
8,Price_to_SMA10,-0.102680,0.102680,Decreases Risk Score,0.030366
9,Mean_Absolute_Return_5D,-0.092090,0.092090,Decreases Risk Score,0.027234


In [ ]:
feature_importance.to_csv(
    "/content/drive/MyDrive/portfolio_risk_feature_importance.csv",
    index=False
)

print("Feature-importance file exported.")

Feature-importance file exported.


In [ ]:
feature_importance.to_csv(
    "/content/drive/MyDrive/portfolio_risk_feature_importance.csv",
    index=False
)

print("Model-driver file exported successfully.")
print("Driver rows:", len(feature_importance))

Model-driver file exported successfully.
Driver rows: 29


In [ ]:
import os

output_folder = (
    "/content/drive/MyDrive/Portfolio Risk Scanner"
)

os.makedirs(
    output_folder,
    exist_ok=True
)

scored_test_data.to_csv(
    output_folder
    + "/portfolio_risk_scored_data.csv",
    index=False
)

model_performance.to_csv(
    output_folder
    + "/portfolio_risk_model_performance.csv",
    index=False
)

feature_importance.to_csv(
    output_folder
    + "/portfolio_risk_feature_importance.csv",
    index=False
)

print("Files saved in:", output_folder)
print("\nFiles inside the folder:")

for filename in os.listdir(output_folder):
    print(filename)

Files saved in: /content/drive/MyDrive/Portfolio Risk Scanner

Files inside the folder:
portfolio_risk_scored_data.csv
portfolio_risk_model_performance.csv
portfolio_risk_feature_importance.csv
